# Station-Level Trend Analysis (Mann-Kendall and Sen's Slope)

The annual mean (H1) can hide stations that behave differently. Here every station with **at least 8 years of data** is tested separately.

- **Mann-Kendall test** (Kendall's tau against year): is there a monotonic trend?
- **Sen's slope** (Theil-Sen estimator): how large is the change in mg/L per year? It is robust against outliers.
- **Benjamini-Hochberg FDR correction (10%)**: many stations are tested at once, so p-values are corrected for multiple testing.

Note: the test statistic is computed with `scipy.stats.kendalltau` (tau-b, corrected for ties).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

In [ ]:
df = pd.read_csv("../data/processed/powerbi_groundwater_nitrate.csv")

df.shape

## Stations with at least 8 years of data

In [ ]:
years_per_station = df.groupby("StationCode")["Year"].nunique()

print("All stations:", len(years_per_station))
print("Stations with >= 8 years:", (years_per_station >= 8).sum())

## Mann-Kendall and Sen's slope per station

In [ ]:
rows = []

for code, g in df.groupby("StationCode"):
    if g["Year"].nunique() >= 8:
        x = g["Year"].values.astype(float)
        y = g["Nitrate_mg_L"].values

        slope, intercept, low, high = stats.theilslopes(y, x)
        tau, p = stats.kendalltau(x, y)

        # constant series give NaN: treat as "no trend"
        rows.append((code, slope, tau, 1.0 if np.isnan(p) else p))

trend = pd.DataFrame(
    rows,
    columns=["StationCode", "SenSlope", "KendallTau", "p"]
)

trend.head()

## Multiple-testing correction (Benjamini-Hochberg, FDR 10%)

Testing 635 stations at the 5% level would produce about 30 false positives by chance alone. The Benjamini-Hochberg procedure controls the expected share of false discoveries among the significant results.

In [ ]:
def benjamini_hochberg(p):
    p = np.asarray(p)
    n = len(p)
    order = np.argsort(p)
    q = p[order] * n / np.arange(1, n + 1)
    q = np.minimum.accumulate(q[::-1])[::-1]
    result = np.empty(n)
    result[order] = np.minimum(q, 1)
    return result


trend["q"] = benjamini_hochberg(trend["p"])

In [ ]:
def classify(row):
    if row["q"] < 0.10 and row["SenSlope"] < 0:
        return "Significant decrease"
    if row["q"] < 0.10 and row["SenSlope"] > 0:
        return "Significant increase"
    return "Not significant"


trend["TrendClass"] = trend.apply(classify, axis=1)

trend["TrendClass"].value_counts()

**Result:** of the 635 stations with at least 8 years of data, 113 show a significant decrease, 36 a significant increase and 486 no significant trend. The decline is real, but uneven.

In [ ]:
print("Stations tested:", len(trend))
print(trend["TrendClass"].value_counts().to_string())

sig = trend[trend["TrendClass"] != "Not significant"]
print("\nMedian Sen's slope of significant decreases:",
      round(sig[sig["SenSlope"] < 0]["SenSlope"].median(), 2), "mg/L per year")
print("Median Sen's slope of significant increases:",
      round(sig[sig["SenSlope"] > 0]["SenSlope"].median(), 2), "mg/L per year")

In [ ]:
counts = trend["TrendClass"].value_counts().reindex(
    ["Significant decrease", "Not significant", "Significant increase"]
)

plt.figure(figsize=(7, 4))
plt.bar(counts.index, counts.values, color=["#2a78d4", "#9a9a94", "#e04848"])

for i, v in enumerate(counts.values):
    plt.text(i, v + 5, str(v), ha="center")

plt.ylabel("Number of stations")
plt.title("Station-level nitrate trends (2012-2022, FDR 10%)")
plt.tight_layout()
plt.show()

## Save the result

In [ ]:
trend.to_csv("../data/processed/station_trends.csv", index=False)